# Purpose

The goal of this notebook is to combine all the model `metrics.csv` into 1 table.

# Import

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Pathing

In [2]:
PROCESSED_PATH = Path("../../data/processed")
MODELING_PATH = PROCESSED_PATH / "modeling"
MODEL_COMPARISON_PATH = PROCESSED_PATH / "model_comparison"

# Create Metric Files

In [3]:
metric_files = {
    "linear_regression": MODELING_PATH / "linear_regression" / "metrics.csv",
    "ridge_regression": MODELING_PATH / "ridge_regression" / "metrics.csv",
    "random_forest": MODELING_PATH / "random_forest" / "metrics.csv",
    "gradient_boosting": MODELING_PATH / "gradient_boosting" / "metrics.csv",
    "garch_spy_only": MODELING_PATH / "garch" / "metrics.csv",
}

# Create Model Comparison

Metrics are loaded from each model's regenerated `metrics.csv` file.

In [4]:
def evaluate_predictions(prediction_path, prediction_col, actual_col="future_volatility_20d"):
    predictions = pd.read_csv(prediction_path)

    y_true = predictions[actual_col].astype(float)
    y_pred = predictions[prediction_col].astype(float)

    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    r2 = 1 - (((y_true - y_pred) ** 2).sum() / ((y_true - y_true.mean()) ** 2).sum())

    return mae, rmse, r2

In [5]:
all_metrics = []

for model_folder, file_path in metric_files.items():
    metrics = pd.read_csv(file_path).rename(columns={
        "test_MAE": "MAE",
        "test_RMSE": "RMSE",
        "test_R2": "R2",
    })

    required_metric_cols = ["model", "MAE", "RMSE", "R2"]
    missing_cols = [col for col in required_metric_cols if col not in metrics.columns]
    if missing_cols:
        raise ValueError(f"{file_path} is missing metric columns: {missing_cols}")

    metrics = metrics[required_metric_cols]
    metrics["source_folder"] = model_folder
    metrics["scope"] = "SPY only" if model_folder == "garch_spy_only" else "All tickers"
    all_metrics.append(metrics)

In [6]:
raw_model_comparison = pd.concat(all_metrics, ignore_index=True)

raw_model_comparison

,model,MAE,RMSE,R2,source_folder,scope
0,Baseline: rolling_volatility_20d,0.004647,0.007130,0.001510,linear_regression,All tickers
1,Linear Regression,0.005107,0.007456,-0.091727,linear_regression,All tickers
2,Baseline: rolling_volatility_20d,0.004647,0.007130,0.001510,ridge_regression,All tickers
3,Linear Regression,0.005107,0.007456,-0.091727,ridge_regression,All tickers
4,Ridge Regression,0.005103,0.007452,-0.090511,ridge_regression,All tickers
5,Baseline: rolling_volatility_20d,0.004647,0.007130,0.001510,random_forest,All tickers
6,Random Forest (tuned),0.004062,0.006517,0.165752,random_forest,All tickers
7,Baseline: rolling_volatility_20d,0.004647,0.007130,0.001510,gradient_boosting,All tickers
8,Gradient Boosting,0.004113,0.006048,0.281682,gradient_boosting,All tickers
9,Baseline: rolling_volatility_20d,0.004018,0.006628,-0.419243,garch_spy_only,SPY only


# Clean All-Ticker Model Comparison

This table keeps only the all-ticker models and removes duplicate baseline or repeated comparison rows.

In [7]:
all_ticker_model_comparison = raw_model_comparison[
    raw_model_comparison["scope"] == "All tickers"
].copy()

all_ticker_model_comparison = all_ticker_model_comparison.drop_duplicates(
    subset=["model"],
    keep="first"
)

all_ticker_model_comparison = all_ticker_model_comparison[
    ["model", "MAE", "RMSE", "R2", "source_folder", "scope"]
].sort_values("RMSE")

all_ticker_model_comparison

,model,MAE,RMSE,R2,source_folder,scope
8,Gradient Boosting,0.004113,0.006048,0.281682,gradient_boosting,All tickers
6,Random Forest (tuned),0.004062,0.006517,0.165752,random_forest,All tickers
0,Baseline: rolling_volatility_20d,0.004647,0.007130,0.001510,linear_regression,All tickers
4,Ridge Regression,0.005103,0.007452,-0.090511,ridge_regression,All tickers
1,Linear Regression,0.005107,0.007456,-0.091727,linear_regression,All tickers


# SPY-Only Statistical Model Comparison

GARCH is shown separately because this notebook only built the statistical model for `SPY`, not every ticker.

In [8]:
garch_spy_comparison = raw_model_comparison[
    raw_model_comparison["scope"] == "SPY only"
].copy()

garch_spy_comparison = garch_spy_comparison[
    ["model", "MAE", "RMSE", "R2", "source_folder", "scope"]
].sort_values("RMSE")

garch_spy_comparison

,model,MAE,RMSE,R2,source_folder,scope
10,"GARCH(1,1)",0.003815,0.006027,-0.173700,garch_spy_only,SPY only
9,Baseline: rolling_volatility_20d,0.004018,0.006628,-0.419243,garch_spy_only,SPY only


# Save Model Comparison Outputs

In [9]:
MODEL_COMPARISON_PATH.mkdir(parents=True, exist_ok=True)

all_ticker_model_comparison.to_csv(
    MODEL_COMPARISON_PATH / "all_ticker_model_comparison_metrics.csv",
    index=False
)

garch_spy_comparison.to_csv(
    MODEL_COMPARISON_PATH / "garch_spy_comparison_metrics.csv",
    index=False
)

raw_model_comparison.to_csv(
    MODEL_COMPARISON_PATH / "raw_model_comparison_metrics.csv",
    index=False
)

print("Saved:", MODEL_COMPARISON_PATH / "all_ticker_model_comparison_metrics.csv")
print("Saved:", MODEL_COMPARISON_PATH / "garch_spy_comparison_metrics.csv")
print("Saved:", MODEL_COMPARISON_PATH / "raw_model_comparison_metrics.csv")

Saved: ..\..\data\processed\model_comparison\all_ticker_model_comparison_metrics.csv
Saved: ..\..\data\processed\model_comparison\garch_spy_comparison_metrics.csv
Saved: ..\..\data\processed\model_comparison\raw_model_comparison_metrics.csv


# Model Comparison Takeaway

For the all-ticker volatility prediction models, Gradient Boosting now has the lowest RMSE and highest R2, so it is currently the strongest predictive model after adding the broader FRED macro features. Random Forest still improves on the baseline, but it no longer ranks first. Linear Regression and Ridge Regression perform almost the same, and both fall below the rolling-volatility baseline after adding the full macro feature set, which suggests the extra correlated macro columns may add noise for simple linear models.

The GARCH model is kept separate because it was only trained and evaluated on `SPY`. It is useful as a statistical volatility benchmark, but it should not be ranked directly against the all-ticker machine learning models.